# Multi armed bandits: Thompson sampling and UCB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/multi_armed_bandits.ipynb)

A self-contained CPU notebook. All data are generated locally; no credentials or repository files are needed.

In [ ]:
%pip -q install numpy scipy scikit-learn matplotlib

## Experiment design

Every policy receives the same per-round potential reward array for each repetition. Only its chosen arm is observed and used to update its state. This pairing reduces comparison noise without giving the learner counterfactual feedback.

The changed scenario reverses arm probabilities after round 1000. Thompson sampling and UCB1 here retain all observations forever. Their standard stationary interpretations do not guarantee adaptation to a changing environment. The plotted bands describe uncertainty in the mean over 100 repetitions, not a prediction interval for one run.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import math


In [ ]:
def bandit_run(method,drift=False,repeats=100,horizon=2000,seed=17):
    env=np.random.default_rng(seed);rng=np.random.default_rng(seed+100)
    probabilities=np.array([.1,.2,.3,.4,.5]);streams=env.random((horizon,repeats,5));rows=np.arange(repeats)
    count=np.zeros((repeats,5));success=np.zeros_like(count);regret=np.zeros((repeats,horizon));chosen=np.zeros((horizon,5));snapshots={}
    for t in range(horizon):
        p=probabilities[::-1] if drift and t>=horizon//2 else probabilities
        if t<5:arm=np.full(repeats,t)
        elif method=='Thompson':arm=rng.beta(1+success,1+count-success).argmax(axis=1)
        elif method=='UCB1':arm=(success/count+np.sqrt(2*np.log(t+1)/count)).argmax(axis=1)
        elif method=='Epsilon 0.1':
            arm=(success/count).argmax(axis=1);explore=rng.random(repeats)<.1;arm[explore]=rng.integers(0,5,size=explore.sum())
        else:raise ValueError(method)
        reward=(streams[t,rows,arm]<p[arm]).astype(float)
        count[rows,arm]+=1;success[rows,arm]+=reward
        regret[:,t]=p.max()-p[arm];chosen[t]=np.bincount(arm,minlength=5)/repeats
        if t+1 in [5,10,20,40,80,160,320,640,1000,1500,2000]:snapshots[t+1]=(1+success[0].copy(),1+count[0]-success[0])
    return {'regret':np.cumsum(regret,axis=1),'chosen':chosen,'snapshots':snapshots,'count':count,'success':success}


In [ ]:
def bandit_experiment():
    runs={};summary={}
    for scenario in ['stationary','changed']:
        runs[scenario]={};summary[scenario]={}
        for name in ['Thompson','UCB1','Epsilon 0.1']:
            run=bandit_run(name,scenario=='changed');runs[scenario][name]=run;final=run['regret'][:,-1]
            summary[scenario][name]={'mean_regret':float(final.mean()),'se':float(final.std(ddof=1)/10),'median_regret':float(np.median(final))}
            assert np.all(run['count'].sum(axis=1)==2000)
            assert np.all(run['success']<=run['count'])
    return summary,runs


In [ ]:
summary, runs = bandit_experiment()
print(summary)
fig, axes = plt.subplots(1,2,figsize=(10,4))
for ax,(scenario,methods) in zip(axes,runs.items()):
    for name,run in methods.items():
        regret=run["regret"]; mean=regret.mean(axis=0)
        se=regret.std(axis=0,ddof=1)/np.sqrt(len(regret))
        ax.plot(np.arange(1,2001),mean,label=name)
        ax.fill_between(np.arange(1,2001),mean-1.96*se,mean+1.96*se,alpha=.15)
    ax.set_title(scenario); ax.set_xlabel("Round"); ax.set_ylabel("Cumulative pseudo-regret")
axes[0].legend(); plt.tight_layout(); plt.show()


## Exercises

1. Replace the prior Beta(1,1) with Beta(2,2), then repeat the stationary comparison.
2. Compare epsilon values .01, .05 and .2 using fresh evaluation streams after any selection.
3. Add a sliding-window policy for the changed environment. Count which observations it retains.
4. Run 1000 repetitions and check whether uncertainty bands shrink approximately with the square root of repetitions.

## Tested environment
Executed on Python 3.12 with NumPy 2.5.3, SciPy 1.18.1, scikit-learn 1.9.1 and Matplotlib 3.11.2. Ranking additionally used XGBoost 3.4.1. Package versions and floating-point libraries can affect timings and fitted results.
